# Fine-tuning IndoBERT — Tier 1 Skor Urgensi (DSS Bansos Bontoramba)

Notebook ini menjalankan **fine-tuning `indobenchmark/indobert-base-p1`** untuk klasifikasi biner
urgensi teks naratif (`rendah` / `tinggi`). Skor urgensi yang dipakai sistem adalah
**probabilitas kelas `tinggi` (0..1)** — keputusan OI-02.

Kode pelatihan **tidak ditulis ulang di sini**: notebook memanggil modul repo
(`ml/tier1/*.py`) agar preprocessing saat latih identik dengan saat inference (tanpa train/serve skew).

**Sebelum menjalankan:** `Runtime → Change runtime type → T4 GPU`.

Alur:
1. Aktifkan GPU & pasang dependensi
2. Ambil kode repo
3. Siapkan data — (a) korpus augmentasi bawaan, atau (b) **CSV teks lokal berlabel** (Fase 6)
4. Fine-tune
5. Evaluasi (Akurasi + F1 + confusion matrix) & kurva pelatihan
6. Uji cepat inference
7. Unduh artefak → taruh di `ml/artifacts/indobert/` pada server

## 1. Cek GPU & pasang dependensi

In [ ]:
!nvidia-smi
!pip -q install "transformers>=4.40" "scikit-learn>=1.4" "Sastrawi>=1.0.1"
import torch
print('torch', torch.__version__, '| cuda tersedia:', torch.cuda.is_available())

## 2. Ambil kode repo

Pilih salah satu: clone dari Git (isi `REPO_URL`), **atau** unggah arsip `ml/` + `app/core` secara manual.
Yang dibutuhkan hanyalah paket `ml/` (dan `app/core/config.py` bila memakai path default).

In [ ]:
import os, pathlib

REPO_URL = ''  # contoh: 'https://github.com/<user>/bansos-dss.git'

if REPO_URL:
    !git clone -q $REPO_URL /content/bansos-dss
else:
    # Fallback: unggah bansos-dss.zip (berisi folder ml/) lewat panel Files, lalu:
    if pathlib.Path('/content/bansos-dss.zip').exists():
        !unzip -q -o /content/bansos-dss.zip -d /content/

os.chdir('/content/bansos-dss')
print('cwd:', os.getcwd())
!ls ml/tier1

## 3. Siapkan data

**Opsi A — korpus augmentasi** (default Fase 2): memvalidasi pipeline sebelum teks lokal siap.

**Opsi B — teks lokal berlabel** (Fase 6, klaim final skripsi): unggah CSV dengan kolom
`teks,label` (`label` ∈ `tinggi|rendah`), set `ASAL_DATA='lokal'` dan arahkan `INPUT_CSV` ke berkas itu.
Jangan unggah data ber-PII ke layanan pihak ketiga tanpa izin tertulis kelurahan (OI-09).

In [ ]:
ASAL_DATA = 'augmentasi'   # 'augmentasi' | 'publik' | 'lokal'
INPUT_CSV = 'data/corpus/tier1_urgensi.csv'

if ASAL_DATA == 'augmentasi':
    !python -m ml.tier1.corpus --n 3200 --seed 42 --out {INPUT_CSV}

!python -m ml.tier1.dataset --input {INPUT_CSV} --outdir data/corpus --test-size 0.2 --seed 42

## 4. Fine-tuning

Hyperparameter awal: `lr=2e-5`, `batch=16`, `epochs=3`, `max_length=128`, warmup 10%, `fp16` aktif.
Catat setiap perubahan di `progres/fase-2-tier1-indobert/README.md`.

In [ ]:
VERSI_MODEL = f'indobert-p1-{ASAL_DATA}-v1'

!python -m ml.tier1.train \
  --train data/corpus/tier1_train.csv \
  --test  data/corpus/tier1_test.csv \
  --outdir ml/artifacts/indobert \
  --epochs 3 --batch-size 16 --lr 2e-5 --max-length 128 --seed 42 --fp16 \
  --asal-data {ASAL_DATA} --versi-model {VERSI_MODEL}

## 5. Evaluasi & kurva pelatihan

In [ ]:
import json, pandas as pd

print(json.dumps(json.load(open('ml/artifacts/indobert/metrics.json', encoding='utf-8')), indent=2, ensure_ascii=False))
kurva = pd.read_csv('ml/artifacts/indobert/kurva_pelatihan.csv')
display(kurva)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(kurva['epoch'], kurva['loss_latih'], marker='o', label='loss latih')
ax[0].plot(kurva['epoch'], kurva['loss_uji'], marker='o', label='loss uji')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].set_title('Kurva loss')
ax[1].plot(kurva['epoch'], kurva['akurasi'], marker='o', label='akurasi')
ax[1].plot(kurva['epoch'], kurva['f1'], marker='o', label='F1 (kelas tinggi)')
ax[1].set_xlabel('epoch'); ax[1].set_ylim(0, 1.02); ax[1].legend(); ax[1].set_title('Metrik data uji')
plt.tight_layout(); plt.savefig('kurva_pelatihan.png', dpi=150); plt.show()

In [ ]:
# Rincian per kategori: memperlihatkan performa pada kasus 'sulit'
# (hard negative = kata urgen dalam konteks negasi; hard positive = urgen tanpa kata kunci).
!python -m ml.tier1.evaluate --model ml/artifacts/indobert --test data/corpus/tier1_test.csv \
  --out ml/artifacts/indobert/evaluasi_rinci.json

## 6. Uji cepat inference (kontrak = keluaran sistem)

In [ ]:
from ml.tier1.infer import UrgencyScorer

scorer = UrgencyScorer('ml/artifacts/indobert')
contoh = [
    'Kepala keluarga menderita sakit kronis dan rumah nyaris roboh saat hujan.',
    'Tidak ada anggota keluarga yang sakit kronis maupun disabilitas.',   # hard negative
    'Penghasilan harian tidak cukup untuk makan tiga kali sehari.',        # hard positive
    'Keluarga memiliki usaha warung kecil yang berjalan lancar.',
]
for teks, hasil in zip(contoh, scorer.score_batch(contoh)):
    print(f'{hasil.skor:.4f}  {teks}')

## 7. Unduh artefak

Ekstrak isi zip ke `ml/artifacts/indobert/` pada server, lalu pastikan
`INDOBERT_MODEL_PATH` (di `.env`) menunjuk ke folder tersebut. `metadata.json` membawa
`versi_model` yang otomatis tersimpan ke kolom `skor_urgensi.versi_model`.

In [ ]:
from google.colab import files

!cp kurva_pelatihan.png ml/artifacts/indobert/ 2>/dev/null
!cd ml/artifacts && zip -qr /content/{VERSI_MODEL}.zip indobert
print('ukuran:'); !du -h /content/{VERSI_MODEL}.zip
files.download(f'/content/{VERSI_MODEL}.zip')